# Part 2 — SQL: verify the numbers and find where no-shows cluster

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")

# Split part2_queries.sql into statements (drop '--' comment lines first)
with open("part2_queries.sql") as f:
    sql_text = "\n".join(line for line in f if not line.strip().startswith("--"))
statements = [s.strip() for s in sql_text.split(";") if s.strip()]
statements = [s for s in statements if not s.upper().startswith(("CREATE DATABASE", "USE "))]
print(len(statements), "statements to run")


def run(stmt):
    """Show a SELECT as a table; execute anything else and report errors."""
    if stmt.upper().startswith(("SELECT", "WITH")):
        return pd.read_sql_query(stmt, conn)
    try:
        conn.execute(stmt)
        print("OK:", stmt.split("(")[0].strip())
    except sqlite3.Error as e:
        print("ERROR:", type(e).__name__, "-", e)


queue = iter(statements)

10 statements to run


## Task 2.1 — Build and load the database

In [2]:
run(next(queue))   # CREATE TABLE clinics
run(next(queue))   # CREATE TABLE appointments

with open("load_data.sql") as f:
    conn.executescript(f.read())
print("load_data.sql executed")

OK: CREATE TABLE clinics
OK: CREATE TABLE appointments
load_data.sql executed


In [3]:
run(next(queue))   # COUNT(*) clinics

,clinics
0,8


In [4]:
run(next(queue))   # COUNT(*), SUM(no_show) appointments

,appointments,no_shows
0,990,218


In [5]:
# Second row with appointment_id 'A0001' -> must fail on the primary key
run(next(queue))

ERROR: IntegrityError - UNIQUE constraint failed: appointments.appointment_id


## Task 2.2 — No-show rate by clinic

In [6]:
# With HAVING COUNT(*) >= 95
run(next(queue))

,clinic_name,city,appointments,no_shows,no_show_pct
0,Hadapsar Care Centre,Pune,143,39,27.3
1,Malviya Nagar Clinic,Jaipur,142,38,26.8
2,Kothrud Family Clinic,Pune,160,36,22.5
3,Vijay Nagar Clinic,Indore,101,21,20.8
4,Edappally Medical Centre,Kochi,142,23,16.2
5,Vaishali Nagar Clinic,Jaipur,130,18,13.8


In [7]:
# Without HAVING: all 8 clinics
all_clinics_sql = run(next(queue))
all_clinics_sql

,clinic_name,city,appointments,no_shows,no_show_pct
0,Gomti Nagar Care Point,Lucknow,90,30,33.3
1,Hadapsar Care Centre,Pune,143,39,27.3
2,Malviya Nagar Clinic,Jaipur,142,38,26.8
3,Kothrud Family Clinic,Pune,160,36,22.5
4,Vijay Nagar Clinic,Indore,101,21,20.8
5,Edappally Medical Centre,Kochi,142,23,16.2
6,Sitabuldi Health Point,Nagpur,82,13,15.9
7,Vaishali Nagar Clinic,Jaipur,130,18,13.8


In [8]:
top = all_clinics_sql.iloc[0]
print(f"Highest rate: {top['clinic_name']} ({top['city']}) at {top['no_show_pct']}% "
      f"but only {top['appointments']} appointments")

Highest rate: Gomti Nagar Care Point (Lucknow) at 33.3% but only 90 appointments


In [9]:
# Reproduce the clinic counts and no-show counts in pandas and compare with SQL
appts = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")

py_counts = (appts.merge(clinics, on="clinic_id")
                  .groupby("clinic_name")
                  .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
                  .reset_index())
py_counts["no_show_pct"] = (100 * py_counts["no_shows"] / py_counts["appointments"]).round(1)

check = py_counts.merge(all_clinics_sql, on="clinic_name", suffixes=("_py", "_sql"))
match = ((check["appointments_py"] == check["appointments_sql"]).all()
         and (check["no_shows_py"] == check["no_shows_sql"]).all())
print("pandas counts match SQL counts exactly:", match)
check[["clinic_name", "appointments_py", "appointments_sql", "no_shows_py", "no_shows_sql"]]

pandas counts match SQL counts exactly: True


,clinic_name,appointments_py,appointments_sql,no_shows_py,no_shows_sql
0,Edappally Medical Centre,142,142,23,23
1,Gomti Nagar Care Point,90,90,30,30
2,Hadapsar Care Centre,143,143,39,39
3,Kothrud Family Clinic,160,160,36,36
4,Malviya Nagar Clinic,142,142,38,38
5,Sitabuldi Health Point,82,82,13,13
6,Vaishali Nagar Clinic,130,130,18,18
7,Vijay Nagar Clinic,101,101,21,21


## Task 2.3 — Does booking far ahead matter?

In [10]:
run(next(queue))

,lead_band,appointments,no_shows,no_show_pct
0,1-7,390,63,16.2
1,8-14,304,53,17.4
2,15-30,253,76,30.0
3,31+,43,26,60.5


## Task 2.4 — Month-by-month trend

In [11]:
run(next(queue))

,month,appointments,no_show_pct,change_pp
0,2025-01,157,24.8,NaN
1,2025-02,151,20.5,-4.3
2,2025-03,142,20.4,-0.1
3,2025-04,177,20.9,0.5
4,2025-05,198,23.7,2.8
5,2025-06,165,21.2,-2.5


## Task 2.5 — Rank clinics inside each city

In [12]:
overall = pd.read_sql_query("SELECT 100.0 * SUM(no_show) / COUNT(*) AS overall_pct FROM appointments", conn)
print("Overall no-show rate:", round(overall.loc[0, "overall_pct"], 2), "%")
run(next(queue))

Overall no-show rate: 22.02 %


,clinic,city,no_show_pct,rank_in_city,vs_overall
0,Vijay Nagar Clinic,Indore,20.8,1,below
1,Malviya Nagar Clinic,Jaipur,26.8,1,above
2,Vaishali Nagar Clinic,Jaipur,13.8,2,below
3,Edappally Medical Centre,Kochi,16.2,1,below
4,Gomti Nagar Care Point,Lucknow,33.3,1,above
5,Sitabuldi Health Point,Nagpur,15.9,1,below
6,Hadapsar Care Centre,Pune,27.3,1,above
7,Kothrud Family Clinic,Pune,22.5,2,above


In [13]:
# every statement in part2_queries.sql has been run
assert next(queue, None) is None
conn.close()